# 🔭 Fink/LSST Alert Viewer

- author: Sylvie Dagoret-Campagne
- helper : Claude Desktop
- creation date : 2026-02-26
- last update : 2026-02-26
  

Visualisation interactive des alertes Rubin/LSST téléchargées via le broker Fink.  
Inspiré du portail [https://lsst.fink-portal.org](https://lsst.fink-portal.org)

**Structure du dataset :**
- `alerts_catalog.parquet` — métadonnées + scores Fink pour toutes les alertes
- `lightcurves/{diaObjectId}.parquet` — courbe de lumière multi-bande par objet
- `cutouts/{diaObjectId}_label{n}.npy` — cutouts (3, H, W) : Science / Template / Difference

**Convention de nommage des colonnes LSST :**
- `r:` = préfixe de la table `diaSource` (schéma LSST DPDD) — ≠ bande spectrale `r`
- `f:` = champs calculés par Fink (classifieurs, cross-matches)
- `r:band` ∈ {`u`, `g`, `r`, `i`, `z`, `y`} — les 6 bandes spectrales de Rubin

Le notebook fink_alert_viewer.ipynb est créé dans ton répertoire scripts/. Il contient 6 cellules :
- Cellule 1 — Setup : imports, palette des 6 bandes Rubin, fonction flux_to_mag
- Cellule 2 — Chargement : catalogue parquet + index des cutouts/lightcurves, résumé du dataset
- Cellule 3 — Sélection : deux variables à modifier :
        pythonTAG   = 'sn_near_galaxy_candidate'   # le tag voulu
        INDEX = 0                             # l'index dans la liste
        Elle affiche la liste des objets du tag avec leur SNR, score SNN et nom TNS si dispo.
- Cellule 4 — Vue rapide (style portail Fink) : 5 panneaux côte à côte — flux, mag AB, Science, Template, Difference — fond sombre comme le portail.
- Cellule 5 — Fiche détaillée 2×3 : flux + mag AB + scores classifieurs en barres horizontales (bleu si > 0.5, rouge sinon) + 3 cutouts avec ZScale, croix centrale et stats min/max/std.
- Cellule 6 — Vue d'ensemble : grille de toutes les vignettes du tag sélectionné (cutout Science + overlay texte), avec l'objet sélectionné encadré en jaune.
- Cellule 7 — Cutouts haute résolution : les 3 cutouts en grand avec cercle d'aperture à 3 pixels et colormap RdBu_r centrée sur 0 pour la Difference.

## Imports and Configuration

Setup : imports, palette des 6 bandes Rubin, fonction flux_to_mag

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from astropy.visualization import ZScaleInterval
import warnings
warnings.filterwarnings('ignore')

# ── Chemins ──────────────────────────────────────────────────────────────────
DATASET_DIR = Path('fink_dataset')
LC_DIR      = DATASET_DIR / 'lightcurves'
CUTOUT_DIR  = DATASET_DIR / 'cutouts'

# ── Palette bandes spectrales Rubin (u g r i z y) ────────────────────────────
BAND_COLORS = {
    'u': '#7B2FBE',   # violet
    'g': '#2CA02C',   # vert
    'r': '#D62728',   # rouge
    'i': '#FF7F0E',   # orange
    'z': '#8C4A2F',   # brun
    'y': '#1A1A1A',   # quasi-noir
}
BAND_WAVELENGTHS = {'u': 365, 'g': 480, 'r': 620, 'i': 750, 'z': 880, 'y': 1000}  # nm

# ── Conversion flux → magnitude AB ───────────────────────────────────────────
# flux en nJy, zeropoint AB uniforme pour toutes les bandes Rubin
RUBIN_ZEROPOINT = 31.4   # mag_AB = -2.5 * log10(flux_nJy) + 31.4

def flux_to_mag(flux, flux_err=None):
    """psfFlux (nJy) → magnitude AB. Retourne NaN pour flux <= 0."""
    flux = np.asarray(flux, dtype=float)
    with np.errstate(invalid='ignore', divide='ignore'):
        mag = np.where(flux > 0, -2.5 * np.log10(flux) + RUBIN_ZEROPOINT, np.nan)
        if flux_err is not None:
            flux_err = np.asarray(flux_err, dtype=float)
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(flux_err / flux), np.nan)
            return mag, mag_err
    return mag

print('✓ Imports OK')

In [ ]:
# Remove to run faster the notebook
# conda install -c conda-forge ipympl
import ipywidgets as widgets
%matplotlib widget

## Read Fink data from catalog

Chargement : catalogue parquet + index des cutouts/lightcurves, résumé du dataset

In [ ]:
# ── Chargement du catalogue ───────────────────────────────────────────────────
catalog = pd.read_parquet(DATASET_DIR / 'alerts_catalog.parquet')

# Index des fichiers cutouts disponibles
cutout_files = {}
for f in CUTOUT_DIR.glob('*.npy'):
    obj_id = int(f.stem.split('_label')[0])
    label  = int(f.stem.split('_label')[1])
    cutout_files[obj_id] = {'path': f, 'label': label}

# Résumé
print(f'Catalogue    : {len(catalog)} alertes, {catalog["r:diaObjectId"].nunique()} objets uniques')
print(f'Cutouts      : {len(cutout_files)} fichiers .npy')
print(f'Light curves : {len(list(LC_DIR.glob("*.parquet")))} fichiers .parquet')
print(f'\nTags disponibles :')
print(catalog.groupby('fink_tag')[['r:diaObjectId', 'label']]
      .agg(n_alertes=('r:diaObjectId','count'), label=('label','first'))
      .to_string())
print(f'\nColonnes du catalogue :')
print([c for c in catalog.columns])

## Select one alert

— Sélection : deux variables à modifier :

        pythonTAG   = 'sn_near_galaxy_candidate'   # le tag voulu
        INDEX = 0                             # l'index dans la liste
        Elle affiche la liste des objets du tag avec leur SNR, score SNN et nom TNS si dispo.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  SÉLECTION DE L'ALERTE À VISUALISER                            ║
# ║  ➜  Modifier TAG et INDEX pour choisir une alerte              ║
# ╚══════════════════════════════════════════════════════════════════╝

# Tags disponibles :
#   'extragalactic_new_candidate'      (label=1)
#   'extragalactic_lt20mag_candidate'  (label=1)
#   'sn_near_galaxy_candidate'         (label=1)
#   'in_tns'                           (label=1)
#   'hostless_candidate'               (label=0)

TAG   = 'sn_near_galaxy_candidate'   # ← choisir le tag
INDEX = 0                             # ← index dans la liste (0 = premier)

# ─────────────────────────────────────────────────────────────────────────────
subset  = catalog[catalog['fink_tag'] == TAG].reset_index(drop=True)
obj_ids = subset['r:diaObjectId'].unique()

print(f'Tag  : {TAG}')
print(f'Objets disponibles dans ce tag ({len(obj_ids)}) :')
for i, oid in enumerate(obj_ids):
    row = subset[subset['r:diaObjectId'] == oid].iloc[0]
    tns = row.get('f:xm_tns_fullname', '')
    tns_str = f" → TNS: {tns}" if pd.notna(tns) and tns else ''
    print(f'  [{i:2d}]  {oid}  band={row["r:band"]}  snr={row["r:snr"]:.1f}  '
          f'snn={row["f:clf_snnSnVsOthers_score"]:.3f}{tns_str}')

## Load selected object

In [ ]:
# ── Chargement des données de l'objet sélectionné ────────────────────────────
selected_id = int(obj_ids[INDEX])
meta        = subset[subset['r:diaObjectId'] == selected_id].iloc[0]

# Courbe de lumière
lc_file = LC_DIR / f'{selected_id}.parquet'
df_lc   = pd.read_parquet(lc_file) if lc_file.exists() else pd.DataFrame()

# Cutouts
cutout_data = cutout_files.get(selected_id)
cutouts_arr = np.load(cutout_data['path']) if cutout_data else None
# shape : (3, H, W)  →  [0]=Science, [1]=Template, [2]=Difference

print(f'diaObjectId : {selected_id}')
print(f'Tag         : {meta["fink_tag"]}  (label={meta["label"]})')
print(f'Coordonnées : RA={meta["r:ra"]:.6f}°  Dec={meta["r:dec"]:.6f}°')
print(f'Bande alerte: {meta["r:band"]}  |  MJD={meta["r:midpointMjdTai"]:.4f}')
print(f'psfFlux     : {meta["r:psfFlux"]:.1f} ± {meta["r:psfFluxErr"]:.1f} nJy  (SNR={meta["r:snr"]:.1f})')
print(f'Classifieurs Fink :')
print(f'  SNN (SN vs others) : {meta["f:clf_snnSnVsOthers_score"]:.4f}')
print(f'  Early SN Ia        : {meta["f:clf_earlySNIa_score"]:.4f}')
print(f'  CATS class/score   : {meta["f:clf_cats_class"]} / {meta["f:clf_cats_score"]:.4f}')
print(f'Cross-matches :')
print(f'  TNS      : {meta.get("f:xm_tns_fullname", "—")}  type={meta.get("f:xm_tns_type", "—")}')
print(f'  SIMBAD   : {meta.get("f:xm_simbad_otype", "—")}')
print(f'  LegacyDR8: zphot={meta.get("f:xm_legacydr8_zphot", "—")}  pstar={meta.get("f:xm_legacydr8_pstar", "—")}')
print(f'  Mangrove : lum_dist={meta.get("f:xm_mangrove_lum_dist", "—")} Mpc')
print(f'Courbe de lumière : {len(df_lc)} mesures')
print(f'Cutouts           : {"OK " + str(cutouts_arr.shape) if cutouts_arr is not None else "manquants"}')
print(f'\nLien portail Fink : https://lsst.fink-portal.org/{selected_id}')

## Quick view
 
     Vue rapide (style portail Fink) : 5 panneaux côte à côte — flux, mag AB, Science, Template, Difference — fond sombre comme le portail.

In [ ]:
# ── Vue d'ensemble style portail Fink ────────────────────────────────────────
# Layout : courbe de lumière (flux) | courbe de lumière (mag) | Science | Template | Difference

fig = plt.figure(figsize=(20, 5))
fig.patch.set_facecolor('#0d1117')
gs  = gridspec.GridSpec(1, 5, figure=fig, wspace=0.3,
                        left=0.05, right=0.97, top=0.88, bottom=0.15)

# ── Titre global ─────────────────────────────────────────────────────────────
tns_str = f"  →  TNS: {meta['f:xm_tns_fullname']}" if pd.notna(meta.get('f:xm_tns_fullname')) and meta.get('f:xm_tns_fullname') else ''
label_str = 'EXTRAGALACTIC' if meta['label'] == 1 else 'OTHER'
fig.suptitle(
    f"diaObjectId {selected_id}{tns_str}     [{label_str}]  |  tag: {meta['fink_tag']}",
    fontsize=11, color='#e6edf3', fontfamily='monospace',
    y=0.97
)

AX_KWARGS = dict(facecolor='#161b22')

# ── 1. Courbe de lumière en flux (nJy) ───────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0], **AX_KWARGS)

if not df_lc.empty and 'r:band' in df_lc.columns:
    t0 = df_lc['r:midpointMjdTai'].min()
    for band in sorted(df_lc['r:band'].unique()):
        mask = df_lc['r:band'] == band
        t    = df_lc.loc[mask, 'r:midpointMjdTai'] - t0
        f    = df_lc.loc[mask, 'r:psfFlux']
        fe   = df_lc.loc[mask, 'r:psfFluxErr']
        ax1.errorbar(t, f, yerr=fe, fmt='o', color=BAND_COLORS.get(band, 'gray'),
                     label=f'band {band}', markersize=5, capsize=2, lw=1)
ax1.axhline(0, color='#444', lw=0.8, ls='--')
ax1.set_xlabel('Δ MJD TAI (days)', color='#8b949e', fontsize=9)
ax1.set_ylabel('psfFlux (nJy)', color='#8b949e', fontsize=9)
ax1.set_title('Light curve — flux', color='#e6edf3', fontsize=10)
ax1.tick_params(colors='#8b949e', labelsize=8)
for spine in ax1.spines.values(): spine.set_edgecolor('#30363d')
ax1.legend(fontsize=7, facecolor='#0d1117', labelcolor='#e6edf3',
           edgecolor='#30363d', markerscale=0.8)

# ── 2. Courbe de lumière en magnitude AB ─────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1], **AX_KWARGS)

if not df_lc.empty and 'r:band' in df_lc.columns:
    t0 = df_lc['r:midpointMjdTai'].min()
    for band in sorted(df_lc['r:band'].unique()):
        mask = df_lc['r:band'] == band
        t    = df_lc.loc[mask, 'r:midpointMjdTai'] - t0
        f    = df_lc.loc[mask, 'r:psfFlux'].values
        fe   = df_lc.loc[mask, 'r:psfFluxErr'].values
        mag, mag_err = flux_to_mag(f, fe)
        valid = np.isfinite(mag)
        if valid.sum() > 0:
            ax2.errorbar(t[valid], mag[valid], yerr=mag_err[valid],
                         fmt='o', color=BAND_COLORS.get(band, 'gray'),
                         label=f'band {band}', markersize=5, capsize=2, lw=1)
ax2.invert_yaxis()
ax2.set_xlabel('Δ MJD TAI (days)', color='#8b949e', fontsize=9)
ax2.set_ylabel('mag AB', color='#8b949e', fontsize=9)
ax2.set_title('Light curve — mag AB', color='#e6edf3', fontsize=10)
ax2.tick_params(colors='#8b949e', labelsize=8)
for spine in ax2.spines.values(): spine.set_edgecolor('#30363d')
ax2.legend(fontsize=7, facecolor='#0d1117', labelcolor='#e6edf3',
           edgecolor='#30363d', markerscale=0.8)

# ── 3-5. Cutouts : Science / Template / Difference ───────────────────────────
cutout_names  = ['Science', 'Template', 'Difference']
cutout_cmaps  = ['gray', 'gray', 'RdBu_r']
zscale = ZScaleInterval(contrast=0.25)

for idx, (name, cmap) in enumerate(zip(cutout_names, cutout_cmaps)):
    ax = fig.add_subplot(gs[0, idx + 2], **AX_KWARGS)
    if cutouts_arr is not None:
        img = cutouts_arr[idx]
        try:
            vmin, vmax = zscale.get_limits(img)
        except Exception:
            vmin, vmax = np.nanpercentile(img, [1, 99])
        # Pour Difference : colormap symétrique centrée sur 0
        if name == 'Difference':
            absmax = max(abs(vmin), abs(vmax))
            vmin, vmax = -absmax, absmax
        im = ax.imshow(img, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax,
                       interpolation='nearest')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                     format='%.0f').ax.tick_params(labelsize=6, colors='#8b949e')
        # Croix au centre
        cy, cx = np.array(img.shape) / 2
        ax.plot(cx, cy, '+', color='#f0e040', ms=12, mew=1.5)
    else:
        ax.text(0.5, 0.5, 'No cutout', transform=ax.transAxes,
                ha='center', va='center', color='#8b949e')
    ax.set_title(name, color='#e6edf3', fontsize=10)
    ax.tick_params(colors='#8b949e', labelsize=7)
    for spine in ax.spines.values(): spine.set_edgecolor('#30363d')
    # Label bande dans le coin
    ax.text(0.03, 0.97, f"band {meta['r:band']}", transform=ax.transAxes,
            color='#f0e040', fontsize=8, va='top', fontfamily='monospace')

plt.savefig(DATASET_DIR / f'{selected_id}_viewer.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print(f'✓ Figure sauvegardée : {DATASET_DIR}/{selected_id}_viewer.png')

## Detailed view

    Fiche détaillée 2×3 : flux + mag AB + scores classifieurs en barres horizontales (bleu si > 0.5, rouge sinon) + 3 cutouts avec ZScale, croix centrale et stats min/max/std.

In [ ]:
# ── Fiche détaillée de l'objet (style portail Fink) ──────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.patch.set_facecolor('#0d1117')
fig.suptitle(f'Fink/LSST  —  diaObjectId {selected_id}',
             fontsize=13, color='#e6edf3', fontfamily='monospace', y=0.98)

# ── Panneau 1 : Flux (toutes bandes) ─────────────────────────────────────────
ax = axes[0, 0]
ax.set_facecolor('#161b22')
if not df_lc.empty:
    t0 = df_lc['r:midpointMjdTai'].min()
    for band in sorted(df_lc['r:band'].unique()):
        mask = df_lc['r:band'] == band
        t  = df_lc.loc[mask, 'r:midpointMjdTai'] - t0
        f  = df_lc.loc[mask, 'r:psfFlux']
        fe = df_lc.loc[mask, 'r:psfFluxErr']
        ax.errorbar(t, f, yerr=fe, fmt='o-', color=BAND_COLORS.get(band,'gray'),
                    label=f'{band} ({BAND_WAVELENGTHS.get(band,"?")}nm)',
                    markersize=4, capsize=2, lw=0.8, alpha=0.9)
ax.axhline(0, color='#444', lw=0.7, ls='--')
ax.set_xlabel('Δ MJD TAI (days)', color='#8b949e', fontsize=9)
ax.set_ylabel('psfFlux (nJy)', color='#8b949e', fontsize=9)
ax.set_title('Light curve — flux (nJy)', color='#58a6ff', fontsize=10)
ax.legend(fontsize=7, facecolor='#0d1117', labelcolor='#c9d1d9', edgecolor='#30363d')
for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
ax.tick_params(colors='#8b949e', labelsize=8)

# ── Panneau 2 : Magnitude AB ──────────────────────────────────────────────────
ax = axes[0, 1]
ax.set_facecolor('#161b22')
if not df_lc.empty:
    t0 = df_lc['r:midpointMjdTai'].min()
    for band in sorted(df_lc['r:band'].unique()):
        mask = df_lc['r:band'] == band
        t    = df_lc.loc[mask, 'r:midpointMjdTai'].values - t0
        f    = df_lc.loc[mask, 'r:psfFlux'].values
        fe   = df_lc.loc[mask, 'r:psfFluxErr'].values
        mag, merr = flux_to_mag(f, fe)
        valid = np.isfinite(mag)
        if valid.sum() > 0:
            ax.errorbar(t[valid], mag[valid], yerr=merr[valid],
                        fmt='o-', color=BAND_COLORS.get(band,'gray'),
                        label=f'{band}', markersize=4, capsize=2, lw=0.8, alpha=0.9)
ax.invert_yaxis()
ax.set_xlabel('Δ MJD TAI (days)', color='#8b949e', fontsize=9)
ax.set_ylabel('mag AB', color='#8b949e', fontsize=9)
ax.set_title('Light curve — mag AB', color='#58a6ff', fontsize=10)
ax.legend(fontsize=7, facecolor='#0d1117', labelcolor='#c9d1d9', edgecolor='#30363d')
for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
ax.tick_params(colors='#8b949e', labelsize=8)

# ── Panneau 3 : Scores classifieurs ──────────────────────────────────────────
ax = axes[0, 2]
ax.set_facecolor('#161b22')
scores = {
    'SNN\n(SN vs others)' : float(meta['f:clf_snnSnVsOthers_score']),
    'Early\nSN Ia'        : max(0, float(meta['f:clf_earlySNIa_score'])),
    'CATS\nscore'         : float(meta['f:clf_cats_score']),
}
labels = list(scores.keys())
vals   = list(scores.values())
colors_bar = ['#58a6ff' if v >= 0.5 else '#f85149' for v in vals]
bars = ax.barh(labels, vals, color=colors_bar, alpha=0.85, height=0.5)
ax.axvline(0.5, color='#f0e040', lw=1.2, ls='--', alpha=0.8, label='threshold 0.5')
ax.set_xlim(0, 1)
for bar, val in zip(bars, vals):
    ax.text(min(val + 0.02, 0.95), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', color='#e6edf3', fontsize=9, fontfamily='monospace')
ax.set_title('Fink classifiers', color='#58a6ff', fontsize=10)
ax.tick_params(colors='#8b949e', labelsize=8)
for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
ax.legend(fontsize=7, facecolor='#0d1117', labelcolor='#c9d1d9', edgecolor='#30363d')

# ── Panneaux 4-6 : Cutouts Science / Template / Difference ───────────────────
cutout_names = ['Science', 'Template', 'Difference']
cutout_cmaps = ['afmhot', 'afmhot', 'RdBu_r']
zscale = ZScaleInterval(contrast=0.25)

for idx, (name, cmap) in enumerate(zip(cutout_names, cutout_cmaps)):
    ax = axes[1, idx]
    ax.set_facecolor('#161b22')
    if cutouts_arr is not None:
        img = cutouts_arr[idx]
        try:
            vmin, vmax = zscale.get_limits(img)
        except Exception:
            vmin, vmax = np.nanpercentile(img, [1, 99])
        if name == 'Difference':
            absmax = max(abs(vmin), abs(vmax))
            vmin, vmax = -absmax, absmax
        im = ax.imshow(img, origin='lower', cmap=cmap,
                       vmin=vmin, vmax=vmax, interpolation='nearest')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                     label='counts').ax.tick_params(labelsize=7, colors='#8b949e')
        # Croix centrale
        cy, cx = np.array(img.shape) / 2
        ax.plot(cx, cy, '+', color='#f0e040', ms=14, mew=1.8)
        # Stats
        ax.text(0.02, 0.02,
                f'min={img.min():.0f}\nmax={img.max():.0f}\nstd={img.std():.0f}',
                transform=ax.transAxes, color='#8b949e', fontsize=6,
                va='bottom', fontfamily='monospace',
                bbox=dict(facecolor='#0d1117', alpha=0.6, edgecolor='none'))
    else:
        ax.text(0.5, 0.5, 'No cutout\navailable',
                transform=ax.transAxes, ha='center', va='center',
                color='#8b949e', fontsize=10)
    ax.set_title(f'Cutout — {name}  [band {meta["r:band"]}]',
                 color='#58a6ff', fontsize=10)
    ax.tick_params(colors='#8b949e', labelsize=7)
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(DATASET_DIR / f'{selected_id}_detail.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'✓ Figure sauvegardée : {DATASET_DIR}/{selected_id}_detail.png')

# Overview over all alert
    
    — Vue d'ensemble : grille de toutes les vignettes du tag sélectionné (cutout Science + overlay texte), avec l'objet sélectionné encadré en jaune.

In [ ]:
# ── Vue d'ensemble : toutes les alertes d'un tag ─────────────────────────────
# Grille de vignettes : cutout Science + mini courbe de lumière

TAG_OVERVIEW = TAG  # ← même tag que la sélection principale

subset_ov = catalog[catalog['fink_tag'] == TAG_OVERVIEW].copy()
oids      = subset_ov['r:diaObjectId'].unique()
n         = len(oids)
ncols     = 5
nrows     = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3.5))
fig.patch.set_facecolor('#0d1117')
fig.suptitle(f'Tag: {TAG_OVERVIEW}  ({n} objets)',
             fontsize=12, color='#e6edf3', fontfamily='monospace', y=1.01)

axes_flat = np.array(axes).flatten()
zscale    = ZScaleInterval(contrast=0.25)

for i, oid in enumerate(oids):
    ax  = axes_flat[i]
    ax.set_facecolor('#161b22')
    row = subset_ov[subset_ov['r:diaObjectId'] == oid].iloc[0]

    # Cutout Science en fond
    cf = cutout_files.get(int(oid))
    if cf is not None:
        arr = np.load(cf['path'])
        img = arr[0]  # Science
        try:
            vmin, vmax = zscale.get_limits(img)
        except Exception:
            vmin, vmax = np.nanpercentile(img, [1, 99])
        ax.imshow(img, origin='lower', cmap='afmhot', vmin=vmin, vmax=vmax,
                  aspect='auto', interpolation='nearest')
        cy, cx = np.array(img.shape) / 2
        ax.plot(cx, cy, '+', color='#f0e040', ms=10, mew=1.5)
    else:
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.text(0.5, 0.5, 'no cutout', ha='center', va='center',
                color='#8b949e', transform=ax.transAxes)

    # Infos en overlay
    snn  = row['f:clf_snnSnVsOthers_score']
    band = row['r:band']
    snr  = row['r:snr']
    tns  = row.get('f:xm_tns_fullname', '')
    tns_str = f"\n{tns}" if pd.notna(tns) and tns else ''
    ax.text(0.02, 0.98,
            f'#{i}  {str(oid)[-6:]}\nSNN={snn:.2f}  SNR={snr:.0f}\nband {band}{tns_str}',
            transform=ax.transAxes, color='#e6edf3', fontsize=6.5,
            va='top', fontfamily='monospace',
            bbox=dict(facecolor='#0d1117', alpha=0.7, edgecolor='none', pad=1.5))

    # Highlight si sélectionné
    if int(oid) == selected_id:
        for sp in ax.spines.values():
            sp.set_edgecolor('#f0e040'); sp.set_linewidth(2.5)
    else:
        for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

# Axes vides
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.savefig(DATASET_DIR / f'overview_{TAG_OVERVIEW}.png', dpi=120,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'✓ Vue d\'ensemble sauvegardée : {DATASET_DIR}/overview_{TAG_OVERVIEW}.png')

## Comparison of cutouts : Science /Template / Difference

    — Cutouts haute résolution : les 3 cutouts en grand avec cercle d'aperture à 3 pixels et colormap RdBu_r centrée sur 0 pour la Difference.

In [ ]:
# ── Comparaison des cutouts : Science / Template / Difference ────────────────
# Vue haute résolution des 3 cutouts côte à côte avec colorbar commune

if cutouts_arr is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5),layout="constrained")
    fig.patch.set_facecolor('#0d1117')
    fig.suptitle(
        f'Cutouts  —  diaObjectId {selected_id}  |  band {meta["r:band"]}  |  '
        f'MJD {meta["r:midpointMjdTai"]:.4f}  |  visit {meta["r:visit"]}',
        fontsize=10, color='#e6edf3', fontfamily='monospace', y=1.01
    )

    names  = ['Science', 'Template', 'Difference']
    cmaps  = ['afmhot', 'afmhot', 'RdBu_r']
    zscale = ZScaleInterval(contrast=0.3)

    for i, (name, cmap) in enumerate(zip(names, cmaps)):
        ax  = axes[i]
        ax.set_facecolor('#161b22')
        img = cutouts_arr[i]
        H, W = img.shape

        try:
            vmin, vmax = zscale.get_limits(img)
        except Exception:
            vmin, vmax = np.nanpercentile(img, [1, 99])
        if name == 'Difference':
            absmax = max(abs(vmin), abs(vmax))
            vmin, vmax = -absmax, absmax

        im = ax.imshow(img, origin='lower', cmap=cmap,
                       vmin=vmin, vmax=vmax, interpolation='nearest',
                       extent=[0, W, 0, H])

        cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cb.set_label('ADU (counts)', color='#8b949e', fontsize=8)
        cb.ax.tick_params(labelsize=7, colors='#8b949e')

        # Croix centrale
        ax.plot(W / 2, H / 2, '+', color='#f0e040', ms=16, mew=2)
        # Cercle d'aperture indicatif (rayon 3 pixels)
        circle = plt.Circle((W/2, H/2), 3, color='#f0e040',
                             fill=False, lw=1, ls='--', alpha=0.7)
        ax.add_patch(circle)

        ax.set_title(name, color='#58a6ff', fontsize=12, pad=8)
        ax.set_xlabel('x (pix)', color='#8b949e', fontsize=9)
        ax.set_ylabel('y (pix)', color='#8b949e', fontsize=9)
        ax.tick_params(colors='#8b949e', labelsize=8)
        for sp in ax.spines.values(): sp.set_edgecolor('#30363d')

        # Stats dans le coin
        ax.text(0.98, 0.02,
                f'{H}×{W} pix\nmin={img.min():.1f}\nmax={img.max():.1f}\nstd={img.std():.1f}',
                transform=ax.transAxes, color='#8b949e', fontsize=7,
                ha='right', va='bottom', fontfamily='monospace',
                bbox=dict(facecolor='#0d1117', alpha=0.7, edgecolor='none', pad=2))

    #plt.tight_layout()
    plt.savefig(DATASET_DIR / f'{selected_id}_cutouts.png', dpi=150,
                bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print(f'✓ Cutouts sauvegardés : {DATASET_DIR}/{selected_id}_cutouts.png')
else:
    print('✗ Cutouts non disponibles pour cet objet.')